In [1]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [2]:
# Importing packages and modules
import comet_ml
import torch
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import CometLogger
from copy import deepcopy
import numpy as np
import gymnasium as gym
from itertools import product
from tqdm import tqdm
from RL4CRN.Input_Output_Rxn_Networks.IOCRN_MassAction import IOCRN_MassAction
from RL4CRN.Environments.CRNEnvironment import CRNEnvironment
from RL4CRN.Environments.VecCRNEnvironment import VecCRNEnvironment
from RL4CRN.Environments.VecCRNEnvironment import SerialVecCRNEnvironment
from RL4CRN.Agents.RecurrentAgent import RecurrentAgent
from RL4CRN.Input_Output_Rxn_Networks.CRNGenerator import CRNCompletor
from RL4CRN.Rewards.Transients import dynamic_tracking_error

In [3]:
# Set the logger to use Comet
api_key = "o77J6VCMDamustkfJuMXZ2jdV"
logger = CometLogger(
    api_key=api_key,
    project="Molecular_Integrators",        
    workspace="maurice-filo" 
)
logger = logger.experiment

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/maurice-filo/molecular-integrators/03b5bf5e651c493fbb714f80b2ac872c



In [4]:
# Construct the template CRN
species_labels = ['X_1', 'Z_1', 'Z_2']
inputs_labels = ['u_1', 'u_2']
stoichiometry_reactants = np.array([[0, 1], [0, 0], [0, 0]], dtype=np.int8)
stoichiometry_products = np.array([[0, 0], [1, 0], [0, 0]], dtype=np.int8)
parameters = np.array([1, 1], dtype=np.float32)
input_influence_matrix = np.array([[1, 0], [0, 1]], dtype=np.int8)
outputs = np.array([1], dtype=np.int8)
CRN_template = IOCRN_MassAction(stoichiometry_reactants, stoichiometry_products, parameters, input_influence_matrix, outputs, species_labels, inputs_labels)
print('CRN template:')
CRN_template.print_reactions()

CRN template:
Inputs: ['u_1', 'u_2'] 
Species: ['X_1', 'Z_1', 'Z_2'] 
Output Species: ['X_1'] 
Reaction 0: 0 -> Z_1 ; Rate Constant: 1.0u_1 
Reaction 1: X_1 -> 0 ; Rate Constant: 1.0u_2 



In [5]:
# Hyperparameters
max_num_reactions = 3                               # Maximum number of reactions
N_CPUs = 128                                        # Number of CPUs          
n_samples = 10*N_CPUs                               # Number of samples    
n_grid = 1
param_lower_bound = 1
param_upper_bound = 1
width = 256
depth = 3
num_species = 3
num_inputs = 2
allow_input_influence = False
learning_rate = 1e-5
entropy_weight = 100
entropy_update_coefficient = 0.75
entropy_schedule = 5
minimum_entropy_weight = 0.1
risk = 0.95
risk_update = 0
maximum_risk = 1.0
risk_schedule = 20
epoch_num = 1000
render_schedule = 5

# Create the reward function
def compute_reward(state):
    nums = [0.5, 1, 1.5]
    u = np.array(list(product(nums, repeat=state.num_inputs)), dtype=np.float32)
    initial_condition = np.array([0, 0, 0], dtype=np.float32)
    time_horizon = np.linspace(0, 300, 1000, dtype=np.float32)
    r = u[:,0] * state.parameters[0]
    return dynamic_tracking_error(state, u, initial_condition, time_horizon, r, threshold=1000)

In [6]:
# Construct parallel environments
CRN_0 = deepcopy(CRN_template)
vec_env = VecCRNEnvironment([CRNEnvironment(CRN_0, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(n_samples)], N_CPUs=N_CPUs, logger=logger)
# vec_env = SerialVecCRNEnvironment([CRNEnvironment(CRN_0, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(n_samples)], logger=logger)

In [7]:
# Construct the agent
device = 'cuda' if torch.cuda.is_available() else 'cpu'
param_grid = np.linspace(param_lower_bound, param_upper_bound, n_grid, dtype=np.float32)
parameter_grid = np.tile(param_grid, (max_num_reactions, 1))
class RSG_Attributes:
    def __init__(self, width, depth, allow_input_influence):
        self.LSTM_hidden_size = width
        self.FFNN_hidden_size = [width, width, width] if allow_input_influence else [width, width]
        self.FFNN_num_layers = [depth, depth, depth] if allow_input_influence else [depth, depth]
        self.weight = [None, None, None] if allow_input_influence else [None, None]
class PSG_Attributes:
    def __init__(self, width, depth):
        self.LSTM_hidden_size = width
        self.FFNN_hidden_size = width
        self.FFNN_num_layers = 3
        self.weight = None

rsg_attributes = RSG_Attributes(width, depth, allow_input_influence)
psg_attributes = PSG_Attributes(width, depth)
completor = CRNCompletor(max_num_reactions, num_species, max_num_reactions, CRN_0.num_unknown_parameters, num_inputs, parameter_grid, n_samples, rsg_attributes, psg_attributes, device=device, allow_input_influence=allow_input_influence).to(device)
agent = RecurrentAgent(vec_env.envs[0], completor, allow_input_influence, logger, learning_rate, entropy_weight, entropy_update_coefficient, entropy_schedule, minimum_entropy_weight, risk, risk_update, maximum_risk, risk_schedule)

In [ ]:
# Training Loop
for i in tqdm(range(epoch_num)):
    vec_env.reset()
    for j in range(max_num_reactions + vec_env.envs[0].CRN_template.num_unknown_parameters):
        actions = agent.act()
        out = vec_env.step(actions)
    rewards = vec_env.get_reward(compute_reward)
    agent.update(rewards)
    if i % render_schedule == 0:
        vec_env.render(rewards, mode='logger_image')

  0%|          | 1/1000 [00:03<58:22,  3.51s/it]